# 06 — Build the final rich EGFR feature table

This notebook combines four information layers:

1. **Baseline local structural features**
2. **Kinase-wide geometric and network features**
3. **Drug and protein–drug interaction features**
4. **Auxiliary EGFR mutation-effect predictions**

It produces:

`egfr_structural_features_rich.csv`

This file will be used to train the final drug-resistance model.

## Inputs

Upload:

- `EGFR_mutant_structure_bundle.zip`
- `egfr_structural_features_enhanced.csv`
- `egfr_auxiliary_model_bundle.zip`

## Important limitation

The mutant structures are generated starting coordinates and have not undergone validated ligand-aware minimization. Hydrogen-bond and clash features are therefore **geometry-based proxies**, not binding free energies.

## Step 1 — Upload the three input files

In [ ]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:")
for name in uploaded:
    print(" -", name)

## Step 2 — Install required packages

In [ ]:
!pip -q install pandas numpy biopython scipy scikit-learn joblib requests

## Step 3 — Extract the two ZIP bundles and load the enhanced feature table

In [ ]:
from pathlib import Path
import zipfile
import shutil
import pandas as pd

structure_zip = "EGFR_mutant_structure_bundle.zip"
auxiliary_zip = "egfr_auxiliary_model_bundle.zip"

structure_dir = Path("EGFR_structure_bundle")
auxiliary_dir = Path("EGFR_auxiliary_bundle")

for directory in [structure_dir, auxiliary_dir]:
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir()

with zipfile.ZipFile(structure_zip, "r") as zf:
    zf.extractall(structure_dir)

with zipfile.ZipFile(auxiliary_zip, "r") as zf:
    zf.extractall(auxiliary_dir)

enhanced = pd.read_csv("egfr_structural_features_enhanced.csv")

print("Enhanced rows:", len(enhanced))
print("Enhanced columns:", len(enhanced.columns))
print("Mutations:", enhanced["mutation"].nunique())
print("Drugs:", sorted(enhanced["drug"].unique()))
display(enhanced.head())

## Step 4 — Obtain drug descriptors from PubChem

The notebook queries PubChem by drug name for:

- molecular weight,
- hydrogen-bond donor count,
- hydrogen-bond acceptor count,
- topological polar surface area,
- rotatable-bond count,
- XLogP.

A small cache is saved in the notebook runtime. Covalent-binding status is added separately.

In [ ]:
import requests
import time

PUBCHEM_PROPERTIES = (
    "MolecularWeight,HydrogenBondDonorCount,HydrogenBondAcceptorCount,"
    "TPSA,RotatableBondCount,XLogP"
)

COVALENT_STATUS = {
    "Erlotinib": 0,
    "Afatinib": 1,
    "Dacomitinib": 1,
    "Osimertinib": 1,
}

def fetch_pubchem_descriptors(drug_name):
    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
        f"{drug_name}/property/{PUBCHEM_PROPERTIES}/JSON"
    )
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    properties = response.json()["PropertyTable"]["Properties"][0]

    return {
        "drug": drug_name,
        "drug_molecular_weight": float(properties["MolecularWeight"]),
        "drug_hbond_donors": float(properties["HBondDonorCount"]),
        "drug_hbond_acceptors": float(properties["HBondAcceptorCount"]),
        "drug_tpsa_A2": float(properties["TPSA"]),
        "drug_rotatable_bonds": float(properties["RotatableBondCount"]),
        "drug_xlogp": float(properties.get("XLogP", float("nan"))),
        "drug_is_covalent": COVALENT_STATUS[drug_name],
    }

drug_rows = []
for drug in sorted(enhanced["drug"].unique()):
    descriptors = fetch_pubchem_descriptors(drug)
    drug_rows.append(descriptors)
    time.sleep(0.2)

drug_descriptors = pd.DataFrame(drug_rows)
display(drug_descriptors)

## Step 5 — Load the structures and define interaction helpers

The following geometry-based features are calculated:

- putative protein–drug polar contacts within 3.5 Å,
- heavy-atom steric clashes within 2.0 Å,
- protein–drug atom contacts within 4.5 Å,
- number of pocket residues contacting the drug,
- local heavy-atom packing around the mutation,
- mutation-side-chain contacts with the drug.

The features are calculated for WT and mutant structures, and differences are recorded.

In [ ]:
import numpy as np
from Bio.PDB import PDBParser
from scipy.spatial.distance import cdist

parser = PDBParser(QUIET=True)

WATER_NAMES = {"HOH", "WAT", "DOD"}
COMMON_IONS = {
    "NA", "K", "CL", "MG", "CA", "ZN", "MN", "FE", "CU", "CO",
    "SO4", "PO4", "GOL", "EDO"
}

def load_first_model(pdb_path):
    structure = parser.get_structure("structure", str(pdb_path))
    return next(structure.get_models())

def heavy_atoms(residue):
    return [atom for atom in residue.get_atoms() if atom.element != "H"]

def atom_coords(atoms):
    return np.array([atom.coord for atom in atoms], dtype=float)

def protein_residues(model):
    for chain in model:
        for residue in chain:
            if residue.id[0] == " ":
                yield chain.id, residue

def protein_atoms(model):
    atoms = []
    for _, residue in protein_residues(model):
        atoms.extend(heavy_atoms(residue))
    return atoms

def find_residue(model, chain_id, position):
    for chain in model:
        if chain.id != chain_id:
            continue
        for residue in chain:
            if residue.id[0] == " " and residue.id[1] == int(position):
                return residue
    raise KeyError(f"Residue {chain_id}:{position} not found")

def identify_ligand(model):
    candidates = []

    for chain in model:
        for residue in chain:
            if residue.id[0] == " ":
                continue

            name = residue.get_resname().strip().upper()
            atoms = heavy_atoms(residue)

            if name in WATER_NAMES or name in COMMON_IONS:
                continue
            if len(atoms) < 5:
                continue

            candidates.append((len(atoms), chain.id, residue))

    if not candidates:
        raise ValueError("No inhibitor-like hetero-residue found")

    candidates.sort(key=lambda item: item[0], reverse=True)
    return candidates[0][1], candidates[0][2]

def distance_matrix(atoms_a, atoms_b):
    if not atoms_a or not atoms_b:
        return np.empty((0, 0))
    return cdist(atom_coords(atoms_a), atom_coords(atoms_b))

def atom_contact_count(atoms_a, atoms_b, cutoff):
    distances = distance_matrix(atoms_a, atoms_b)
    if distances.size == 0:
        return 0
    return int((distances <= cutoff).sum())

def polar_atoms(atoms):
    return [atom for atom in atoms if atom.element in {"N", "O", "S"}]

def putative_polar_contacts(protein_atoms_list, ligand_atoms, cutoff=3.5):
    return atom_contact_count(
        polar_atoms(protein_atoms_list),
        polar_atoms(ligand_atoms),
        cutoff,
    )

def steric_clashes(protein_atoms_list, ligand_atoms, cutoff=2.0):
    return atom_contact_count(protein_atoms_list, ligand_atoms, cutoff)

def pocket_residue_ids(model, ligand_atoms, cutoff=4.5):
    contacting = set()

    for chain_id, residue in protein_residues(model):
        distances = distance_matrix(heavy_atoms(residue), ligand_atoms)
        if distances.size and distances.min() <= cutoff:
            contacting.add((chain_id, residue.id[1]))

    return contacting

def local_packing_count(model, target_residue, radius=6.0):
    target_sidechain = [
        atom for atom in heavy_atoms(target_residue)
        if atom.name not in {"N", "CA", "C", "O"}
    ]

    if not target_sidechain:
        target_sidechain = heavy_atoms(target_residue)

    nearby = 0
    target_ids = {id(atom) for atom in heavy_atoms(target_residue)}

    for atom in protein_atoms(model):
        if id(atom) in target_ids:
            continue
        distances = distance_matrix(target_sidechain, [atom])
        if distances.size and distances.min() <= radius:
            nearby += 1

    return nearby

def sidechain_atoms(residue):
    atoms = [
        atom for atom in heavy_atoms(residue)
        if atom.name not in {"N", "CA", "C", "O"}
    ]
    return atoms if atoms else heavy_atoms(residue)

## Step 6 — Calculate rich interaction-change features

In [ ]:
reference_dir = structure_dir / "reference_structures"
mutant_dir = structure_dir / "mutant_structures"

interaction_rows = []

for row in enhanced.itertuples(index=False):
    wt_path = reference_dir / f"{row.pdb_id}.pdb"

    mutant_name = f"{row.drug}_{row.pdb_id}_{row.mutation}.pdb"
    mutant_path = mutant_dir / mutant_name

    if not mutant_path.exists():
        raise FileNotFoundError(f"Missing mutant structure: {mutant_path}")

    wt_model = load_first_model(wt_path)
    mutant_model = load_first_model(mutant_path)

    wt_residue = find_residue(wt_model, row.chain, row.position)
    mutant_residue = find_residue(mutant_model, row.chain, row.position)

    _, wt_ligand = identify_ligand(wt_model)
    _, mutant_ligand = identify_ligand(mutant_model)

    wt_ligand_atoms = heavy_atoms(wt_ligand)
    mutant_ligand_atoms = heavy_atoms(mutant_ligand)

    wt_protein_atoms = protein_atoms(wt_model)
    mutant_protein_atoms = protein_atoms(mutant_model)

    wt_pocket = pocket_residue_ids(wt_model, wt_ligand_atoms)
    mutant_pocket = pocket_residue_ids(mutant_model, mutant_ligand_atoms)

    wt_polar = putative_polar_contacts(wt_protein_atoms, wt_ligand_atoms)
    mutant_polar = putative_polar_contacts(
        mutant_protein_atoms, mutant_ligand_atoms
    )

    wt_clashes = steric_clashes(wt_protein_atoms, wt_ligand_atoms)
    mutant_clashes = steric_clashes(
        mutant_protein_atoms, mutant_ligand_atoms
    )

    wt_atom_contacts = atom_contact_count(
        wt_protein_atoms, wt_ligand_atoms, 4.5
    )
    mutant_atom_contacts = atom_contact_count(
        mutant_protein_atoms, mutant_ligand_atoms, 4.5
    )

    wt_packing = local_packing_count(wt_model, wt_residue, 6.0)
    mutant_packing = local_packing_count(
        mutant_model, mutant_residue, 6.0
    )

    wt_sidechain_drug_contacts = atom_contact_count(
        sidechain_atoms(wt_residue), wt_ligand_atoms, 4.5
    )
    mutant_sidechain_drug_contacts = atom_contact_count(
        sidechain_atoms(mutant_residue), mutant_ligand_atoms, 4.5
    )

    interaction_rows.append({
        "drug": row.drug,
        "pdb_id": row.pdb_id,
        "mutation": row.mutation,

        "wt_putative_polar_contacts_3p5A": wt_polar,
        "mut_putative_polar_contacts_3p5A": mutant_polar,
        "putative_polar_contacts_lost":
            max(0, wt_polar - mutant_polar),
        "putative_polar_contacts_gained":
            max(0, mutant_polar - wt_polar),

        "wt_steric_clashes_2A": wt_clashes,
        "mut_steric_clashes_2A": mutant_clashes,
        "delta_steric_clashes_2A":
            mutant_clashes - wt_clashes,

        "wt_protein_drug_atom_contacts_4p5A": wt_atom_contacts,
        "mut_protein_drug_atom_contacts_4p5A": mutant_atom_contacts,
        "delta_protein_drug_atom_contacts_4p5A":
            mutant_atom_contacts - wt_atom_contacts,

        "wt_pocket_residue_count_4p5A": len(wt_pocket),
        "mut_pocket_residue_count_4p5A": len(mutant_pocket),
        "pocket_residues_lost_4p5A": len(wt_pocket - mutant_pocket),
        "pocket_residues_gained_4p5A": len(mutant_pocket - wt_pocket),

        "wt_local_packing_atoms_6A": wt_packing,
        "mut_local_packing_atoms_6A": mutant_packing,
        "delta_local_packing_atoms_6A":
            mutant_packing - wt_packing,

        "wt_sidechain_drug_contacts_4p5A":
            wt_sidechain_drug_contacts,
        "mut_sidechain_drug_contacts_4p5A":
            mutant_sidechain_drug_contacts,
        "delta_sidechain_drug_contacts_4p5A":
            mutant_sidechain_drug_contacts
            - wt_sidechain_drug_contacts,
    })

interaction_features = pd.DataFrame(interaction_rows)

print("Interaction rows:", len(interaction_features))
display(interaction_features.head())

## Step 7 — Load the auxiliary model and calculate an auxiliary score for each Hayes mutation

The auxiliary model was trained on the large EGFR mutation-effect dataset. Its prediction is used as one additional feature; it is not treated as the resistance label.

In [ ]:
import json
import joblib
import re

aux_model_path = (
    auxiliary_dir / "auxiliary_mutation_effect_model.joblib"
)
aux_dfi_path = auxiliary_dir / "egfr_dfi_profile.csv"

aux_model = joblib.load(aux_model_path)
aux_dfi = pd.read_csv(aux_dfi_path)
dfi_lookup = aux_dfi.set_index("ResI").to_dict("index")

AA_VOLUME = {
    "A": 88.6, "R": 173.4, "N": 114.1, "D": 111.1, "C": 108.5,
    "Q": 143.8, "E": 138.4, "G": 60.1, "H": 153.2, "I": 166.7,
    "L": 166.7, "K": 168.6, "M": 162.9, "F": 189.9, "P": 112.7,
    "S": 89.0, "T": 116.1, "W": 227.8, "Y": 193.6, "V": 140.0,
}

AA_HYDROPATHY = {
    "A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5,
    "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2, "I": 4.5,
    "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8, "P": -1.6,
    "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2,
}

AA_CHARGE = {
    "A": 0.0, "R": 1.0, "N": 0.0, "D": -1.0, "C": 0.0,
    "Q": 0.0, "E": -1.0, "G": 0.0, "H": 0.1, "I": 0.0,
    "L": 0.0, "K": 1.0, "M": 0.0, "F": 0.0, "P": 0.0,
    "S": 0.0, "T": 0.0, "W": 0.0, "Y": 0.0, "V": 0.0,
}

AA_AROMATIC = {
    aa: int(aa in {"F", "W", "Y", "H"}) for aa in AA_VOLUME
}
AA_POLAR = {
    aa: int(
        aa in {"R", "N", "D", "Q", "E", "H", "K", "S", "T", "Y", "C"}
    )
    for aa in AA_VOLUME
}

def parse_mutation(mutation):
    match = re.fullmatch(r"([A-Z])(\d+)([A-Z])", mutation.strip().upper())
    if not match:
        raise ValueError(f"Unsupported mutation: {mutation}")
    wt, position, mutant = match.groups()
    return wt, int(position), mutant

def auxiliary_input_row(mutation):
    wt, position, mutant = parse_mutation(mutation)
    dfi_row = dfi_lookup.get(position, {})

    return {
        "wt_aa": wt,
        "mut_aa": mutant,
        "position": position,
        "position_scaled": (position - 712) / (978 - 712),
        "delta_sidechain_volume_A3":
            AA_VOLUME[mutant] - AA_VOLUME[wt],
        "absolute_volume_change_A3":
            abs(AA_VOLUME[mutant] - AA_VOLUME[wt]),
        "delta_hydropathy":
            AA_HYDROPATHY[mutant] - AA_HYDROPATHY[wt],
        "absolute_hydropathy_change":
            abs(AA_HYDROPATHY[mutant] - AA_HYDROPATHY[wt]),
        "delta_charge":
            AA_CHARGE[mutant] - AA_CHARGE[wt],
        "charge_changed":
            int(AA_CHARGE[mutant] != AA_CHARGE[wt]),
        "delta_aromatic":
            AA_AROMATIC[mutant] - AA_AROMATIC[wt],
        "delta_polar":
            AA_POLAR[mutant] - AA_POLAR[wt],
        "site_DFI": dfi_row.get("DFI", np.nan),
        "site_DFI_percentile": dfi_row.get("DFI%", np.nan),
        "site_rigidity_score":
            dfi_row.get("minus_ln_pctdfi", np.nan),
    }

unique_mutations = sorted(enhanced["mutation"].unique())
aux_inputs = pd.DataFrame(
    [auxiliary_input_row(mutation) for mutation in unique_mutations],
    index=unique_mutations,
)

auxiliary_scores = pd.DataFrame({
    "mutation": unique_mutations,
    "auxiliary_predicted_mutation_effect":
        aux_model.predict(aux_inputs),
    "mutation_site_DFI": aux_inputs["site_DFI"].values,
    "mutation_site_DFI_percentile":
        aux_inputs["site_DFI_percentile"].values,
    "mutation_site_rigidity_score":
        aux_inputs["site_rigidity_score"].values,
})

display(auxiliary_scores)

## Step 8 — Merge all four feature layers

The final table retains the experimental targets:

- `auc_ratio_vs_wt`
- `resistant`

and adds the drug, interaction, DFI, and auxiliary-model features.

In [ ]:
rich = enhanced.merge(
    drug_descriptors,
    on="drug",
    how="left",
    validate="many_to_one",
)

rich = rich.merge(
    interaction_features.drop_duplicates(
        subset=["drug", "pdb_id", "mutation"]
    ),
    on=["drug", "pdb_id", "mutation"],
    how="left",
    validate="many_to_one",
)

rich = rich.merge(
    auxiliary_scores,
    on="mutation",
    how="left",
    validate="many_to_one",
)

assert len(rich) == len(enhanced)

added_columns = [column for column in rich.columns if column not in enhanced.columns]

print("Final rows:", len(rich))
print("Final columns:", len(rich.columns))
print("\nAdded columns:")
for column in added_columns:
    print(" -", column)

display(rich.head())

## Step 9 — Sanity checks

In [ ]:
print("Missing values among added features:")
missing = rich[added_columns].isna().sum()
display(missing[missing > 0].sort_values(ascending=False))

print("\nAdded-feature ranges:")
display(rich[added_columns].describe().T)

print("\nRows by drug:")
display(rich.groupby("drug").size().rename("rows"))

## Step 10 — Save and download the final rich feature table

This file becomes the input for the final resistance-model comparison.

In [ ]:
output_file = "egfr_structural_features_rich.csv"
rich.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Shape:", rich.shape)

files.download(output_file)

## Next step

Use the same grouped training framework to compare:

1. baseline local features,
2. kinase-wide features,
3. rich interaction + drug + DFI + auxiliary features.

The final comparison must use identical mutation-grouped cross-validation splits so that differences reflect feature information rather than easier data splitting.